# Dynamic Gesture Recognition for ISL

This notebook focuses on recognizing dynamic gestures in Irish Sign Language (ISL), specifically for the letters J, X, and Z which require motion.

## Setup and Dependencies

In [ ]:
!pip install opencv-python mediapipe numpy matplotlib pandas scikit-learn tensorflow

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from google.colab import drive
from tqdm.notebook import tqdm

# Initialize MediaPipe solutions
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

## Mount Google Drive

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Set the path to your dataset
DATASET_PATH = '/content/drive/MyDrive/ISL-HS'

## Functions for Processing Videos

In [ ]:
def extract_hand_landmarks_from_frame(frame):
    """
    Extract hand landmarks from a single frame using MediaPipe.
    
    Args:
        frame: Input video frame
        
    Returns:
        List of landmarks or None if no hand detected
    """
    # Convert the BGR frame to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Process the frame and detect hands
    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:
        results = hands.process(frame_rgb)
    
    # Check if hand landmarks were detected
    if not results.multi_hand_landmarks:
        return None
    
    # Extract landmarks
    landmarks = []
    for landmark in results.multi_hand_landmarks[0].landmark:
        landmarks.append([landmark.x, landmark.y, landmark.z])
    
    return np.array(landmarks).flatten()

def process_video(video_path, max_frames=30):
    """
    Process a video file to extract hand landmarks for each frame.
    
    Args:
        video_path: Path to the video file
        max_frames: Maximum number of frames to process
        
    Returns:
        Sequence of hand landmarks
    """
    cap = cv2.VideoCapture(video_path)
    frames_landmarks = []
    
    frame_count = 0
    while cap.isOpened() and frame_count < max_frames:
        success, frame = cap.read()
        if not success:
            break
            
        landmarks = extract_hand_landmarks_from_frame(frame)
        
        if landmarks is not None:
            frames_landmarks.append(landmarks)
        else:
            # If no hand detected, use zeros
            frames_landmarks.append(np.zeros(63))  # 21 landmarks with x,y,z
            
        frame_count += 1
    
    cap.release()
    
    # Pad or truncate to ensure consistent sequence length
    if len(frames_landmarks) < max_frames:
        # Pad with zeros
        padding = [np.zeros(63) for _ in range(max_frames - len(frames_landmarks))]
        frames_landmarks.extend(padding)
    elif len(frames_landmarks) > max_frames:
        # Truncate
        frames_landmarks = frames_landmarks[:max_frames]
    
    return np.array(frames_landmarks)

def load_dynamic_gestures(dataset_path, letters=['J', 'X', 'Z'], max_frames=30):
    """
    Load and process videos for dynamic gestures.
    
    Args:
        dataset_path: Path to the dataset
        letters: List of dynamic gesture letters to process
        max_frames: Maximum number of frames per video
        
    Returns:
        X: Sequences of landmarks
        y: Labels
    """
    X = []
    y = []
    
    for letter in tqdm(letters, desc="Processing dynamic gestures"):
        letter_path = os.path.join(dataset_path, letter)
        
        if not os.path.exists(letter_path):
            print(f"Path not found: {letter_path}")
            continue
            
        for filename in os.listdir(letter_path):
            if filename.lower().endswith(('.mov', '.mp4', '.avi')):
                video_path = os.path.join(letter_path, filename)
                sequence = process_video(video_path, max_frames)
                
                X.append(sequence)
                y.append(letter)
    
    return np.array(X), np.array(y)

## Load and Process Dynamic Gesture Data

In [ ]:
# Load dynamic gesture data
X, y = load_dynamic_gestures(DATASET_PATH)

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

## Build and Train LSTM Model for Dynamic Gestures

In [ ]:
# Build LSTM model for sequence classification
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.5),
    LSTM(32),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(y_categorical.shape[1], activation='softmax')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

## Evaluate the Model

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_accuracy:.4f}")

# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## Save the Model

In [ ]:
# Save the model
model.save('/content/drive/MyDrive/isl_dynamic_gesture_model.h5')

# Save the label encoder
import pickle
with open('/content/drive/MyDrive/dynamic_label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

## Test with a Sample Video

In [ ]:
def predict_dynamic_gesture(video_path):
    """
    Predict the dynamic gesture from a video.
    
    Args:
        video_path: Path to the video
        
    Returns:
        Predicted letter and confidence
    """
    # Process the video
    sequence = process_video(video_path)
    
    # Reshape for prediction
    sequence = sequence.reshape(1, sequence.shape[0], sequence.shape[1])
    
    # Make prediction
    prediction = model.predict(sequence)
    predicted_class = np.argmax(prediction)
    confidence = prediction[0][predicted_class]
    
    # Get the letter
    predicted_letter = label_encoder.inverse_transform([predicted_class])[0]
    
    return predicted_letter, confidence

# Test with a sample video
# Replace with a path to your test video
test_video_path = '/content/drive/MyDrive/test_video.mp4'
letter, confidence = predict_dynamic_gesture(test_video_path)
print(f"Predicted letter: {letter}, Confidence: {confidence:.4f}")

# Display a few frames from the video
cap = cv2.VideoCapture(test_video_path)
frames = []
for i in range(5):  # Get 5 frames
    ret, frame = cap.read()
    if ret:
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()

plt.figure(figsize=(15, 3))
for i, frame in enumerate(frames):
    plt.subplot(1, 5, i+1)
    plt.imshow(frame)
    plt.axis('off')
    if i == 2:  # Middle frame
        plt.title(f"Predicted: {letter}\nConfidence: {confidence:.4f}")
plt.tight_layout()
plt.show()

## Next Steps

1. Integrate with the static gesture recognition model
2. Implement real-time recognition from webcam
3. Export the model for use in the FastAPI application
4. Improve model performance with data augmentation